In [10]:
import os
import openai, json

client = openai.OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))
messages = [
    {
        "role": "system",
        "content": ("You are a helpful assistant that can use tools to answer questions."
        ),
    }
]

In [8]:
def get_weather(city):
    return f"The weather in {city} is 25°C."

FUNCTION_MAP= {
    'get_weather': get_weather
}

In [4]:
TOOLS = [
  {
    "type" : "function",
    "function" : {
      "name" : "get_weather",
      "description" : "A function to get the weather of a city.",
      "parameters" : {
        "type" : "object",
        "properties" : {
          "city" : {
            "type" : "string",
            "description" : "The name of the city to get the weather of."
          }
        },
        "required" : ["city"]
      }
    }
  }
]

In [5]:
from openai.types.chat import ChatCompletionMessage

def call_ai():
  response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=TOOLS,
  )
  process_ai_response(response.choices[0].message)


def process_ai_response(message: ChatCompletionMessage): 
  if message.tool_calls:
      messages.append({
          "role": "assistant", 
          "content": message.content or "",
          "tool_calls": [
              {
                  "id": tc.id,
                  "type": "function",
                  "function": {
                      "name": tc.function.name,
                      "arguments": tc.function.arguments
                  }
              } 
              for tc in message.tool_calls
          ]
      })
      
      for tool_call in message.tool_calls:
          function_name = tool_call.function.name
          arguments = tool_call.function.arguments

          print(f"Calling function {function_name} with {arguments}")

          try: 
            arguments = json.loads(arguments)
          except json.JSONDecodeError:
            arguments = {}

          function_to_run = FUNCTION_MAP.get(function_name)
          result = function_to_run(**arguments)

          messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
          })
      call_ai()

  else:
    messages.append({"role": "assistant", "content": message.content or ""})
    print(f"Assistant: {message.content}")

In [7]:
while True:
    message = input("Send a message to the LLM... ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: how is the weather of Spain? 


BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_3IyeJAEvDY93uve8H6Vo2Ejq", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}